# 같은 12프레임에서 YOLO가 체간의 변화를 읽는가?

검증 선수 6명의 이륙과 무릎 통과에서 사전학습 YOLO26n-pose의 가까운 쪽 어깨와 골반을 두 번의 수동 판독과 비교합니다. 22번 실험의 프레임과 각도식을 고정하고, 모델은 학습하지 않습니다. **Run All**로 실행하세요.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ultralytics import YOLO

from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS
from wlcv.evaluation.trunk_model import (
    candidate_gallery, compare_frames, plot_changes, plot_trial, predict_frames, trial_changes,
)

split = pd.read_csv(SPLIT)
validation = split.loc[split["role"] == "validation", ["video_id", "person_id", "group"]]
metadata = pd.read_csv(ROOT / "data/metadata/videos.csv")
videos = validation.merge(metadata[["video_id", "filename", "near_side"]], on="video_id",
                          validate="one_to_one").set_index("video_id")
folder = ANNOTATIONS / "trunk_inclination"
weights = ROOT / "models/yolo26n-pose.pt"
if not weights.is_file():
    raise FileNotFoundError(f"로컬 자세 모델 가중치를 확인하세요. {weights}")
model = YOLO(str(weights))

## 1. 원본에서 수행자 박스 확인

CPU, 입력 640, 사람 박스 기준 0.25는 기존 자세 베이스라인과 같습니다. 여러 박스가 나오면 아래 번호와 원본을 먼저 보세요. 같은 선수의 일부만 잡힌 박스와 전신 박스가 겹치면 전신을 포함한 후보를 선택합니다.

In [ ]:
results = predict_frames(model, videos, folder, VIDEOS)
candidate_gallery(results)
plt.show()

## 2. 가까운 쪽의 두 관절과 수동점 비교

b02의 30번은 상체만 포함한 1번과 전신을 포함한 2번 후보가 겹칩니다. 다른 조건에서 다시 실행하면 위의 후보 번호를 재확인하세요. 다른 사람만 잡힌 프레임은 해당 키의 값을 `None`으로 두세요. 양쪽 관절을 섞지 않도록 메타데이터의 선수 기준 좌우를 사용합니다.

어깨와 골반이 모두 기존 표시 기준 0.50 이상이어야 모델 각도를 비교합니다. 좌표가 있지만 저신뢰이면 ‘후보 각도’에만 남습니다. 각도는 골반에서 어깨로 향하는 선과 화면 수직선의 사이각입니다. 수동 골반은 두 회차 모두 소켓 추정점이므로 모델과의 차이는 참고값입니다.

In [ ]:
choices = {("b02_sn_h_02", 30): 2}
frames, joints = compare_frames(results, videos, folder, choices)
display(frames.style.format(precision=2, na_rep="—"))

## 3. 이륙에서 무릎 통과까지 얼마나 달라졌나?

무릎 통과 각도에서 이륙 각도를 뺍니다. 양수면 더 기울고, 음수면 더 세워졌습니다. 수동 두 회차의 방향이 다르면 모델의 방향 일치율 계산에서 제외합니다. 변화 방향이 일치해도 크기는 다를 수 있으므로 각 회차와의 차이도 함께 봅니다.

In [ ]:
trials = trial_changes(frames)
display(trials.style.format(precision=2, na_rep="—"))
print(trials["방향 판정"].value_counts().to_string())
plot_changes(trials)
plt.show()

## 4. 어깨와 골반 중 어느 점이 달랐나?

`person`을 바꾸면 같은 선수의 두 시점을 나란히 볼 수 있습니다. 분홍 원은 모델점, 두 종류의 십자 표식은 각 회차의 수동점입니다. 사진에 점이 겹쳐 보이면 아래 표에서 원본 기준 거리와 신뢰도를 확인하세요. 선수마다 원본 해상도가 다르므로 픽셀 거리는 같은 프레임의 점끼리 비교하세요.

In [ ]:
person = "b02"
plot_trial(person, frames, joints, results)
plt.show()
display(joints.loc[joints["선수"] == person].style.format(precision=2, na_rep="—"))

## 5. 결과를 로컬에 저장

프레임별 각도, 관절 좌표·신뢰도, 선수별 변화를 CSV로 저장합니다. 그래프와 여섯 선수의 비교 그림은 같은 결과 폴더에 저장됩니다.

In [ ]:
output = ROOT / "outputs/pose_baseline/trunk_model"
output.mkdir(parents=True, exist_ok=True)
for name, table in [("frames", frames), ("joints", joints), ("trials", trials)]:
    table.to_csv(output / f"{name}.csv", index=False)

fig = plot_changes(trials)
fig.savefig(output / "changes.png", dpi=135)
plt.close(fig)
for person_id in trials["선수"]:
    fig = plot_trial(person_id, frames, joints, results)
    fig.savefig(output / f"{person_id}.png", dpi=135)
    plt.close(fig)
print(output)